## Cleaning 02_nav_history.csv

In [1]:
import pandas as pd

df = pd.read_csv("../data/raw/02_nav_history.csv")
print(df['date'].dtype)

str


In [2]:
# convert date to datetime
df['date'] = pd.to_datetime(df['date'])


In [3]:
# Sort by amfi_code + date
df = df.sort_values(["amfi_code", "date"])

In [4]:
# Forward fill NAV

df['nav'] = df.groupby('amfi_code')['nav'].ffill()


In [5]:
# Remove duplicates
df = df.drop_duplicates()

In [6]:
# Keep only valid NAV values
df = df[df['nav'] > 0]

In [7]:
# Save cleaned file
df.to_csv(
    "../data/processed/02_nav_history_clean.csv",
    index=False
)


## Cleaning 08_investor_transactions.csv

In [8]:
import pandas as pd
df_tx = pd.read_csv("../data/raw/08_investor_transactions.csv")
df_tx.head()
df_tx.info()
df_tx['transaction_date'].dtype


<class 'pandas.DataFrame'>
RangeIndex: 32778 entries, 0 to 32777
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   investor_id         32778 non-null  str    
 1   transaction_date    32778 non-null  str    
 2   amfi_code           32778 non-null  int64  
 3   transaction_type    32778 non-null  str    
 4   amount_inr          32778 non-null  int64  
 5   state               32778 non-null  str    
 6   city                32778 non-null  str    
 7   city_tier           32778 non-null  str    
 8   age_group           32778 non-null  str    
 9   gender              32778 non-null  str    
 10  annual_income_lakh  32778 non-null  float64
 11  payment_mode        32778 non-null  str    
 12  kyc_status          32778 non-null  str    
dtypes: float64(1), int64(2), str(10)
memory usage: 5.4 MB


<StringDtype(na_value=nan)>

In [9]:
df_tx['transaction_date'] = pd.to_datetime(df_tx['transaction_date'])
df_tx['transaction_date'].dtype


dtype('<M8[us]')

In [10]:
# Standardise Transaction Types

df_tx['transaction_type'] = (
    df_tx['transaction_type']
    .astype(str)
    .str.strip()
    .str.lower()
)

# Map SIP, Lumpsum, Redemption properly
type_map = { 'sip':'SIP','lumpsum':'Lumpsum','redemption':'Redemption'}

df_tx['transaction_type'] = df_tx['transaction_type'].map(type_map).fillna(df_tx['transaction_type'])
#Verify
df_tx['transaction_type'].unique()


<ArrowStringArray>
['SIP', 'Redemption', 'Lumpsum']
Length: 3, dtype: str

In [11]:
#  Validate amount_inr > 0
df_tx = df_tx[df_tx["amount_inr"] > 0]


In [12]:
# Validate KYC Status

df_tx['kyc_status'] = (df_tx['kyc_status'].astype(str).str.strip().str.title())

valid_kyc = [ 
    'Verified',
    'Pending',
    'Failed']

df_tx = df_tx[df_tx['kyc_status'].isin(valid_kyc)]
df_tx['kyc_status'].unique()




<ArrowStringArray>
['Verified', 'Pending']
Length: 2, dtype: str

In [13]:
#  Fix date formats
df_tx["transaction_date"] = pd.to_datetime(df_tx["transaction_date"])
df_tx["transaction_date"] = df_tx["transaction_date"].dt.strftime("%Y-%m-%d")

In [14]:
# Remove Duplicates
df.duplicated().sum()
df_tx = df_tx.drop_duplicates()
df_tx.isnull().sum()
df_tx = df_tx.dropna()

In [15]:
# Save Cleaned File
df_tx.to_csv( "../data/processed/08_investor_transactions_clean.csv",index=False)

## Clean 07_scheme_performance.csv

In [25]:
import pandas as pd

df_per = pd.read_csv(
    "../data/raw/07_scheme_performance.csv"
)

df_per.head()
print(df_per.columns.tolist())

['amfi_code', 'scheme_name', 'fund_house', 'category', 'plan', 'return_1yr_pct', 'return_3yr_pct', 'return_5yr_pct', 'benchmark_3yr_pct', 'alpha', 'beta', 'sharpe_ratio', 'sortino_ratio', 'std_dev_ann_pct', 'max_drawdown_pct', 'aum_crore', 'expense_ratio_pct', 'morningstar_rating', 'risk_grade']


In [17]:

# Convert to Numeric
df_per['return_1yr_pct'] = pd.to_numeric(
    df_per['return_1yr_pct'],
    errors='coerce'
)

In [18]:
#Apply to All Return Columns
return_cols = [
    'return_1yr_pct',
    'return_3yr_pct',
    'return_5yr_pct'
]

for col in return_cols:
    df_per[col] = pd.to_numeric(
        df_per[col],
        errors='coerce'
    )

In [19]:
#Find Missing Returns
df_per[return_cols].isnull().sum()
df_per = df_per.dropna(
    subset=return_cols
)

In [20]:
# Flag Anomalies
df_per['anomaly_flag'] = (
    (df_per['return_1yr_pct'] > 100)
    |
    (df_per['return_1yr_pct'] < -50)
)

In [21]:
# Validate Expense Ratio
invalid_expense = df_per[
    (df_per['expense_ratio_pct'] < 0.1)
    |
    (df_per['expense_ratio_pct'] > 2.5)
]

df_per = df_per[
    (df_per['expense_ratio_pct'] >= 0.1)
    &
    (df_per['expense_ratio_pct'] <= 2.5)
]

In [22]:
# Check Duplicates
df_per.duplicated().sum()
df_per = df_per.drop_duplicates()


In [23]:
# Check Missing Values
df_per.isnull().sum()
df = df.dropna()

In [24]:
# Save Cleaned File
df_per.to_csv(
    "../data/processed/07_scheme_performance_clean.csv",
    index=False
)